<a href="https://colab.research.google.com/github/douglaskorvo/tourism_supply_chain/blob/main/TSSC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.[...]

# Tourism service supply chains — reproducible analysis

**Configuration, reported operational disruption and customer dissatisfaction in 11,509 tourism providers across ten emerging economies**

Companion notebook to the manuscript submitted to *Sustainability* (MDPI), Special Issue *Sustainable Management of Logistics and Supply Chain*.

Douglas S. Rodrigues — Production Engineering Department, Fluminense Federal University

---

This notebook reproduces the full pipeline reported in Section 2 of the manuscript: cleaning, dictionary-based construct coding, measurement diagnostics, model estimation, robustness analyses[...]

## 1. Environment

The cell below prints the exact software statement reported in Section 2.5 of the manuscript. Re-run it before submission so that the reported versions match the environment actually used.

In [ ]:
import sys, platform
import pandas as pd, numpy as np, statsmodels, scipy, matplotlib

print(f"Python {platform.python_version()} ({sys.platform})")
for m in (pd, np, statsmodels, scipy, matplotlib):
    print(f"  {m.__name__:12s} {m.__version__}")

print("\n--- Sentence for Section 2.5 ---")
print(f"All analyses were conducted in Python {'.'.join(platform.python_version_tuple()[:2])} "
      f"using pandas {pd.__version__}, NumPy {np.__version__}, statsmodels {statsmodels.__version__}, "
      f"SciPy {scipy.__version__} and Matplotlib {matplotlib.__version__}.")

Python 3.12.13 (linux)
  pandas       2.2.2
  numpy        2.0.2
  statsmodels  0.14.6
  scipy        1.16.3
  matplotlib   3.10.0

--- Sentence for Section 2.5 ---
All analyses were conducted in Python 3.12 using pandas 2.2.2, NumPy 2.0.2, statsmodels 0.14.6, SciPy 1.16.3 and Matplotlib 3.10.0.


In [ ]:
import re, unicodedata, warnings
from pathlib import Path
import statsmodels.formula.api as smf
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 40)

# ---------------------------------------------------------------- configuration
CFG = dict(
    raw_csv   = Path("reviews_raw.csv"),   # <-- point to the raw export
    outdir    = Path("output"),
    seed      = 20260816,
    fe        = "C(city)",                 # primary specification: city fixed effects
    fe_sens   = "C(country)",              # sensitivity: country fixed effects
    drop_ambiguous = True,                 # exclude providers retrieved under both strata
)
CFG["outdir"].mkdir(exist_ok=True)
np.random.seed(CFG["seed"])

# publication figure style
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.linewidth": .8, "figure.dpi": 150, "savefig.dpi": 300,
                     "savefig.bbox": "tight"})
C_SHORT, C_LONG, C_NEUT = "#2E5E8C", "#C1663A", "#555555"
print("configuration loaded")

configuration loaded


## 2. Data loading and cleaning

Corresponds to Sections 2.2 and 2.3. Every exclusion is counted so that the stepwise totals reported in Table S1 are generated by the code rather than transcribed by hand.

In [ ]:
LOG = []
def step(label, n):
    LOG.append({"step": label, "records": n})
    print(f"{label:<52s} n = {n:>6,}")

raw = pd.read_csv(CFG["raw_csv"])
step("0. Raw export", len(raw))
print(f"   provider identifiers: {raw.place_id.nunique():,}")

df = raw.copy()
for c in ["place_id", "country", "city", "segment", "author_name", "time_desc", "lang"]:
    df[c] = df[c].astype(str).str.strip()

def clean_text(s):
    '''Unicode NFC, strip control characters, collapse whitespace and blank-line runs.'''
    if not isinstance(s, str):
        return np.nan
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"\(Traduzido pelo Google\)"... (truncated)

0. Raw export                                        n = 54,687
   provider identifiers: 12,145
